# Parallelising Robot Evaluation with a Process Pool

Evaluating fitness is almost always the bottleneck in an evolutionary robotics run: every robot has to be simulated before selection can proceed, and the simulations are independent of each other, which makes them easy to run in parallel.

This notebook shows how to spread robot evaluation over the CPU cores with `ariel.simulation.process_pool`, a `concurrent.futures.ProcessPoolExecutor` that ARIEL sets up for MuJoCo. It needs no extra dependency.

The pattern is:

1. Put the function that scores one robot in a `.py` file beside the notebook (here `locomotion_eval.py`).
2. Open the pool in a `with` block: `with process_pool(workers=8) as pool:`.
3. Call `pool.map(function, jobs)`: it returns one result per job, in the order of the jobs.

## Part 1: Serial Baseline

This is what a robot evaluation loop looks like without parallelisation. Run it first to understand the structure before the parallel version is introduced.

### Imports

In [1]:
import time

import numpy as np
import torch
from rich.console import Console

from ariel.body_phenotypes.robogen_lite.constructor import (
    construct_mjspec_from_graph,
)
from ariel.body_phenotypes.robogen_lite.decoders.hi_prob_decoding import (
    HighProbabilityDecoder,
)
from ariel.ec import Individual, Population
from ariel.ec.genotypes.nde import NeuralDevelopmentalEncoding
from ariel.simulation.environments import SimpleFlatWorld

console = Console()

### Encoding setup

We use a `NeuralDevelopmentalEncoding` to map a genotype vector to a robot morphology graph, then `HighProbabilityDecoder` to extract the graph, and `construct_mjspec_from_graph` to build the MuJoCo XML.

In [2]:
NUM_MODULES = 10
GENE_SIZE = 64
SEED = 42

rng = np.random.default_rng(SEED)

nde = NeuralDevelopmentalEncoding(number_of_modules=NUM_MODULES, genotype_size=GENE_SIZE)
hpd = HighProbabilityDecoder(num_modules=NUM_MODULES)

SPAWN = (0.0, 0.0, 0.1)
TARGET = (2.0, 0.0, 0.1)


def genotype_to_xml(genotype: list[list[float]]) -> str:
    """Convert a raw genotype to a MuJoCo XML string (world + robot)."""
    tensor = torch.tensor(genotype, dtype=torch.float32)
    p_mats = nde.forward(tensor)
    graph = hpd.probability_matrices_to_graph(p_mats[0], p_mats[1], p_mats[2])
    spec_obj = construct_mjspec_from_graph(graph)
    world = SimpleFlatWorld()
    world.spawn(spec_obj.spec, position=SPAWN, correct_collision_with_floor=True)
    return world.spec.to_xml()


def create_individual() -> Individual:
    ind = Individual()
    ind.genotype = rng.normal(loc=0, scale=64, size=(3, GENE_SIZE)).tolist()
    return ind

### Fitness function

`evaluate_robot` in `locomotion_eval.py` scores one robot: it optimises a small ANN controller with CMA-ES (10 generations of 5) and returns the final distance to the target (lower is better). Its job is the robot's MuJoCo XML and a seed. Part 2 explains why it lives in a file and not in a cell.

In [3]:
from locomotion_eval import evaluate_robot

### Run, serial

In [4]:
POP_SIZE = 20

population = Population([create_individual() for _ in range(POP_SIZE)])
xml_strings = [genotype_to_xml(ind.genotype) for ind in population]
jobs = [(xml, SEED + i) for i, xml in enumerate(xml_strings)]

t0 = time.perf_counter()
fitnesses_serial = [evaluate_robot(job) for job in jobs]
elapsed_serial = time.perf_counter() - t0

console.print(f"Serial | best={min(fitnesses_serial):.3f} | mean={np.mean(fitnesses_serial):.3f} | time={elapsed_serial:.1f}s")

Serial | best=1.974 | mean=1.996 | time=66.6s

## Part 2: Adding Parallelisation

The changes are confined to the evaluation step. Every other operator (selection, crossover, mutation) stays the same.

| | Serial | Parallel |
|---|---|---|
| Extra import | none | `from ariel.simulation import process_pool` |
| Fitness function | anywhere | in a `.py` file, at its top level |
| Evaluation loop | `[evaluate_robot(job) for job in jobs]` | `list(pool.map(evaluate_robot, jobs))` |
| Process management | none | `with process_pool(workers=8) as pool:` |

### Step 1, keep the fitness function in a `.py` file

`process_pool` starts every worker with the `spawn` method, on every platform: each worker is a fresh Python interpreter that imports the function it runs by its module name. A function defined in a notebook cell lives in the kernel's `__main__`, which a worker cannot import, so the pool would fail to unpickle it. A function in a `.py` file beside the notebook works, because the worker can import that file. That is why `evaluate_robot` lives in `locomotion_eval.py`.

In [5]:
from ariel.simulation import process_pool

### Step 2, replace the `for` loop with `pool.map`

```python
# BEFORE (serial)
fitnesses = [evaluate_robot(job) for job in jobs]

# AFTER (parallel)
with process_pool(workers=NUM_WORKERS) as pool:
    fitnesses = list(pool.map(evaluate_robot, jobs))
```

`pool.map` hands the jobs to the workers as they come free and returns the results in the order of the jobs, so `fitnesses[i]` belongs to `jobs[i]` as before.

## Part 3: Run the Parallel Version

In [6]:
NUM_WORKERS = 8

t0 = time.perf_counter()
with process_pool(workers=NUM_WORKERS) as pool:
    fitnesses_parallel = list(pool.map(evaluate_robot, jobs))
elapsed_parallel = time.perf_counter() - t0

console.print(f"Parallel | best={min(fitnesses_parallel):.3f} | mean={np.mean(fitnesses_parallel):.3f} | time={elapsed_parallel:.1f}s")
console.print(f"Speedup  | {elapsed_serial / elapsed_parallel:.1f}x")

Parallel | best=1.956 | mean=1.996 | time=20.3s

Speedup  | 3.3x

The two runs give slightly different fitness values because `_Network` draws its starting weights from torch's unseeded generator. With a fully seeded fitness function the parallel results equal the serial ones: the slow CPG notebooks in `tests/functional/simulation` print the same distances to the last digit either way.

## Part 4: Tips and Common Pitfalls

**Put the fitness function in a `.py` file, not a notebook cell.**  
Workers import it by module name and cannot see the kernel's `__main__`. Move it to a file beside the notebook and import it; jobs and results must be picklable too (numbers, arrays, strings, pydantic models).

**Open one pool for many generations.**  
Starting a pool takes a few seconds, because each worker imports torch, MuJoCo and ARIEL (4 to 8 s for 1 to 16 workers on an 8-core machine). Open it once around the whole evolution loop, not once per generation.

**Use about one worker per core.**  
`process_pool()` without `workers` starts one per logical CPU. On a machine with two threads per core, the second thread adds little: 256 gecko trials took 6.0 times less time with 8 workers and 7.2 times less with 16, on 8 cores.

**Threading is already handled.**  
Each worker runs torch on one thread. Without that, 8 workers with 8 threads each would compete for the same cores and run slower than serial.

**Do not use threads instead.**  
A MuJoCo control callback (`mujoco.set_mjcb_control`) is one global per process, so trials that set it cannot share a process.

**Only parallelise evaluation.**  
Selection, crossover and mutation work on the shared population in the main process.